# FingerBall v2 Stage 2B — Manual anchor, bidirectional tracking, WiLoR 21-joint QC
Colab: https://colab.research.google.com/github/atstuyuki/FingerBall-Analysis/blob/main/notebooks/FingerBall_v2_Stage2B_Anchored_Tracking_21Landmarks.ipynb

21点の2D表示は未検証の暫定投影です。正規のWiLoR前処理・カメラ式が検証されるまで定量解析には使わないでください。


## 0. 環境セットアップ

In [ ]:
%pip -q install "openvino>=2025,<2027" "mediapipe==0.10.35" "opencv-python-headless>=4.8" matplotlib pandas pyyaml tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. 設定

In [ ]:
from pathlib import Path
import os,time,json,shutil,urllib.request
import numpy as np,pandas as pd,cv2,openvino as ov,mediapipe as mp,yaml
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from google.colab import files
PROJECT_NAME='母指回内トラッキング'
PROJECT_PATH=''
VIDEO_PATH=''
THROWING_HAND='right'
YOLO_CLASS_RIGHT=1
CONF_MIN=0.10
POSE_VIS_MIN=0.35
WRIST_GATE=1.4
FLOW_MAX_AGE=12
FLOW_MIN_VALID=0.45
STRIDE=1
RUN_WILOR=True
CACHE=Path('/content/fingerball_stage2a_cache')
RESULTS_NAME='FingerBall_v2_stage2b_results'
POSE_TASK_URL='https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker/float16/latest/pose_landmarker_full.task'
assert THROWING_HAND in ('right','left') and STRIDE>=1
print('OpenVINO:',ov.__version__,'MediaPipe:',mp.__version__)

## 2. Google Drive上の既存モデルをコピー

In [ ]:
def find_project():
    if PROJECT_PATH:
        p=Path(PROJECT_PATH)
        if not p.is_dir():raise FileNotFoundError(p)
        return p
    for root in [Path('/content/drive/MyDrive'),Path('/content/drive/Shareddrives')]:
        if (root/PROJECT_NAME).is_dir():return root/PROJECT_NAME
        if root.name=='Shareddrives' and root.is_dir():
            for d in root.iterdir():
                if (d/PROJECT_NAME).is_dir():return d/PROJECT_NAME
    raise FileNotFoundError('PROJECT_PATHを指定してください')
project=find_project()
a=project/'thumb_pronation/out/bench_light/verify/int8_leftcal'
b=project/'models/_DATA/wilor/pretrained_models/detector_openvino_model_288x512'
sources=[a/'wilor_core_int8_leftcal.xml',a/'wilor_core_int8_leftcal.bin',b/'detector.xml',b/'detector.bin',b/'metadata.yaml']
if not all(p.is_file() for p in sources):raise FileNotFoundError([(str(p),p.is_file()) for p in sources])
CACHE.mkdir(exist_ok=True,parents=True)
for src in sources:
    folder=CACHE/('wilor' if 'wilor_core' in src.name else 'detector')
    folder.mkdir(exist_ok=True)
    dest=folder/src.name
    if not dest.exists() or dest.stat().st_size!=src.stat().st_size:shutil.copy2(src,dest)
meta=yaml.safe_load((CACHE/'detector/metadata.yaml').read_text(encoding='utf8'))
print('Source:',project,'Hand classes:',meta.get('names'))

## 3. MediaPipe Pose Task / OpenVINOを準備

In [ ]:
task = CACHE / 'pose_landmarker_full.task'
POSE_TASK_URLS = [
    'https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/latest/pose_landmarker_full.task',
    'https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/1/pose_landmarker_full.task',
]
# 壊れた/途中までのファイルは再取得
if task.exists() and task.stat().st_size < 1000000:
    task.unlink()
if not task.is_file():
    errors = []
    for url in POSE_TASK_URLS:
        tmp = task.with_suffix('.download')
        try:
            print('MediaPipe Poseモデル取得:', url)
            request = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(request, timeout=90) as response, open(tmp, 'wb') as out:
                shutil.copyfileobj(response, out)
            if tmp.stat().st_size < 1000000:
                raise ValueError(f'不完全なモデル: {tmp.stat().st_size} bytes')
            tmp.replace(task)
            print('取得成功:', round(task.stat().st_size / 1e6, 1), 'MB')
            break
        except Exception as e:
            errors.append(f'{url}: {type(e).__name__}: {e}')
            tmp.unlink(missing_ok=True)
    if not task.is_file():
        raise RuntimeError('Poseモデルを取得できません。\\n' + '\\n'.join(errors))
assert task.stat().st_size > 1000000, 'Poseモデルのファイルサイズが不正です'
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
po=mp_vision.PoseLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=str(task)),
    running_mode=mp_vision.RunningMode.VIDEO,num_poses=1,
    min_pose_detection_confidence=.4,min_pose_presence_confidence=.4,min_tracking_confidence=.4)
pose=mp_vision.PoseLandmarker.create_from_options(po)
core=ov.Core()
det=core.compile_model(core.read_model(str(CACHE/'detector/detector.xml')),'CPU')
wilor=core.compile_model(core.read_model(str(CACHE/'wilor/wilor_core_int8_leftcal.xml')),'CPU')
dims=list(det.input(0).shape)
assert len(dims)==4 and dims[1]==3,dims
net_h,net_w=map(int,dims[2:])
nc=len(meta.get('names',{0:'left',1:'right'}))
print('Detector:',dims,'WiLoR inputs:',[list(p.shape) for p in wilor.inputs])
print('WiLoR outputs:',[list(p.shape) for p in wilor.outputs])

## 4. 動画を読み込む

In [ ]:
if not VIDEO_PATH:
    uploaded=files.upload()
    if not uploaded:raise RuntimeError('動画未選択')
    VIDEO_PATH=str(Path('/content')/next(iter(uploaded)))
video_src=Path(VIDEO_PATH)
if not video_src.is_file():raise FileNotFoundError(video_src)
source=video_src
if str(video_src).startswith('/content/drive/'):
    local=CACHE/('input_'+video_src.name)
    if not local.exists() or local.stat().st_size!=video_src.stat().st_size:shutil.copy2(video_src,local)
    source=local
cap=cv2.VideoCapture(str(source))
if not cap.isOpened():raise RuntimeError('OpenCVで動画を開けません')
total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT));fps=float(cap.get(cv2.CAP_PROP_FPS))
W=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH));H=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
ok,im=cap.read();cap.release()
if not ok:raise RuntimeError('動画が読めません')
print('Movie:',video_src.name,'frames:',total,'fps:',fps,'image:',W,H)
plt.figure(figsize=(10,5));plt.imshow(cv2.cvtColor(im,cv2.COLOR_BGR2RGB));plt.axis('off');plt.show()

## 5. 検出・腕ROI・LK光学フローの関数

出力の source は yolo_pose / yolo_flow / flow / pose_fallback / none。YOLOとROIの一致が得られた場合だけ yolo_validated=1 とします。

In [ ]:
def clip_box(b):
    if b is None:return None
    a=np.asarray(b,float).copy()
    if a.shape!=(4,) or not np.isfinite(a).all():return None
    a[[0,2]]=np.clip(a[[0,2]],0,W-1);a[[1,3]]=np.clip(a[[1,3]],0,H-1)
    return a if a[2]-a[0]>=8 and a[3]-a[1]>=8 else None
def ctr(b):return (b[:2]+b[2:])/2
def boxside(b):return max(b[2]-b[0],b[3]-b[1])
def rect(x,y,s):return clip_box([x-s/2,y-s/2,x+s/2,y+s/2])
def nms(boxes,scores,thr=.7):
    if not len(boxes):return []
    boxes=np.asarray(boxes);idx=np.argsort(-np.asarray(scores));keep=[]
    while len(idx):
        i=int(idx[0]);keep.append(i);rest=idx[1:]
        if not len(rest):break
        mn=np.maximum(boxes[i,:2],boxes[rest,:2]);mx=np.minimum(boxes[i,2:],boxes[rest,2:])
        intersection=np.prod(np.maximum(mx-mn,0),axis=1)
        a=np.prod(np.maximum(boxes[i,2:]-boxes[i,:2],0))
        b=np.prod(np.maximum(boxes[rest,2:]-boxes[rest,:2],0),axis=1)
        idx=rest[intersection/(a+b-intersection+1e-8)<=thr]
    return keep
def letterbox(img):
    h,w=img.shape[:2];scale=min(net_h/h,net_w/w)
    nw,nh=int(round(w*scale)),int(round(h*scale))
    dx,dy=(net_w-nw)/2,(net_h-nh)/2
    l,rr=int(round(dx-.1)),int(round(dx+.1))
    t,bb=int(round(dy-.1)),int(round(dy+.1))
    z=cv2.resize(img,(nw,nh),interpolation=cv2.INTER_LINEAR)
    z=cv2.copyMakeBorder(z,t,bb,l,rr,cv2.BORDER_CONSTANT,value=(114,114,114))
    return np.ascontiguousarray(z[None,...,::-1].transpose(0,3,1,2)).astype(np.float32)/255.,scale,l,t
def detect_hands(img):
    x,scale,l,t=letterbox(img)
    raw=np.asarray(det([x])[det.output(0)])
    if raw.ndim!=3 or raw.shape[0]!=1:raise RuntimeError('YOLO tensor shape: '+str(raw.shape))
    p=raw[0]
    if p.shape[0]>=4+nc:p=p.T
    if p.shape[1]<4+nc:raise RuntimeError('YOLO channels: '+str(p.shape))
    scores=p[:,4:4+nc]
    target=YOLO_CLASS_RIGHT if THROWING_HAND=='right' else 1-YOLO_CLASS_RIGHT
    conf=scores[:,target]
    k=conf>=CONF_MIN
    z=p[k,:4];cc=conf[k]
    if not len(z):return [],float(conf.max())
    boxes=np.column_stack([z[:,0]-z[:,2]/2,z[:,1]-z[:,3]/2,z[:,0]+z[:,2]/2,z[:,1]+z[:,3]/2])
    out=[]
    for i in nms(boxes,cc):
        b=boxes[i].copy()
        b[[0,2]]=(b[[0,2]]-l)/scale;b[[1,3]]=(b[[1,3]]-t)/scale
        b=clip_box(b)
        if b is not None:out.append((b,float(cc[i])))
    return out,float(conf.max())
def pose_arm(img,t_ms):
    image=mp.Image(image_format=mp.ImageFormat.SRGB,data=cv2.cvtColor(img,cv2.COLOR_BGR2RGB))
    result=pose.detect_for_video(image,int(t_ms))
    if not result.pose_landmarks:return None
    landmarks=result.pose_landmarks[0]
    ids=(12,14,16) if THROWING_HAND=='right' else (11,13,15)
    v=[float(landmarks[i].visibility) for i in ids]
    if min(v)<POSE_VIS_MIN:return None
    sh,el,wr=[np.array([landmarks[i].x*W,landmarks[i].y*H]) for i in ids]
    fore=float(np.linalg.norm(el-wr))
    if not np.isfinite(fore) or fore<12:return None
    return {'shoulder':sh,'elbow':el,'wrist':wr,'forearm':fore,'visibility':min(v)}
def flow_box(prev_gray,gray,b):
    if prev_gray is None or b is None:return None,0.
    bb=np.rint(b).astype(int)
    x0,y0,x1,y1=bb
    mask=np.zeros((H,W),np.uint8);mask[max(0,y0):min(H,y1),max(0,x0):min(W,x1)]=255
    p0=cv2.goodFeaturesToTrack(prev_gray,maxCorners=80,qualityLevel=.01,minDistance=4,mask=mask)
    if p0 is None or len(p0)<6:return None,0.
    p1,s1,_=cv2.calcOpticalFlowPyrLK(prev_gray,gray,p0,None,winSize=(21,21),maxLevel=3)
    if p1 is None:return None,0.
    p2,s2,_=cv2.calcOpticalFlowPyrLK(gray,prev_gray,p1,None,winSize=(21,21),maxLevel=3)
    if p2 is None:return None,0.
    fb=np.linalg.norm((p2-p0).reshape(-1,2),axis=1)
    valid=(s1[:,0]>0)&(s2[:,0]>0)&(fb<1.5)
    frac=float(valid.mean())
    if valid.sum()<6 or frac<FLOW_MIN_VALID:return None,frac
    shift=np.median((p1-p0).reshape(-1,2)[valid],axis=0)
    return clip_box(b+np.tile(shift,2)),frac
def patch_for_wilor(img,box):
    # Stage2A近似切り出し: オリジナルのガウシアン前処理との一致は未検証
    cx,cy=ctr(box);s=2.5*boxside(box)
    if THROWING_HAND=='left':
        img=cv2.flip(img,1);cx=W-1-cx
    M=np.array([[256/s,0,128-cx*256/s],[0,256/s,128-cy*256/s]],np.float32)
    return cv2.warpAffine(img,M,(256,256),flags=cv2.INTER_LINEAR).astype(np.float32)


## 6. 任意フレームで手動BOXを指定

まずフレーム番号を設定し、表示画像を見ながら `MANUAL_BOX=(x0,y0,x1,y1)` を入力してください。Colabでのマウス選択も可能です（下のセル）。

In [ ]:
MANUAL_FRAME=180
MANUAL_BOX=None
RELEASE_START,RELEASE_END=175,210
assert 0<=MANUAL_FRAME<total
cc=cv2.VideoCapture(str(source))
cc.set(cv2.CAP_PROP_POS_FRAMES,MANUAL_FRAME)
ok,anchor_img=cc.read();cc.release()
if not ok:raise RuntimeError("Anchor frame unavailable")
plt.figure(figsize=(12,6))
plt.imshow(cv2.cvtColor(anchor_img,cv2.COLOR_BGR2RGB))
plt.title(f"Manually select pitching hand — frame {MANUAL_FRAME}")
plt.axis("on");plt.show()
if MANUAL_BOX is None:
    import base64
    from IPython.display import display,HTML
    from google.colab.output import eval_js
    ok,buf=cv2.imencode(".jpg",anchor_img)
    dat=base64.b64encode(buf.tobytes()).decode("ascii")
    html = """
    <p>画像上の投球手をドラッグして矩形を描いてください。選択すると実行が進みます。</p>
    <canvas id="fb-anchor" style="max-width:100%;cursor:crosshair;border:1px solid gray"></canvas>
    <div id="fb-result">Drag to select</div>
    <script>
    (() => {
      const c=document.getElementById('fb-anchor'),ctx=c.getContext('2d'),img=new Image();
      window.fbSelectedBox=null;
      img.onload=()=>{c.width=img.width;c.height=img.height;ctx.drawImage(img,0,0)};
      img.src="data:image/jpeg;base64,"+__IMAGE__;
      const xy=e=>{const r=c.getBoundingClientRect();return [(e.clientX-r.left)*c.width/r.width,(e.clientY-r.top)*c.height/r.height]};
      let p=null;
      c.onpointerdown=e=>{p=xy(e);c.setPointerCapture(e.pointerId)};
      c.onpointermove=e=>{if(!p)return;const q=xy(e);ctx.drawImage(img,0,0);ctx.strokeStyle="lime";ctx.lineWidth=3;ctx.strokeRect(p[0],p[1],q[0]-p[0],q[1]-p[1])};
      c.onpointerup=e=>{if(!p)return;const q=xy(e);const b=[Math.min(p[0],q[0]),Math.min(p[1],q[1]),Math.max(p[0],q[0]),Math.max(p[1],q[1])];p=null;
        if(b[2]-b[0]>8&&b[3]-b[1]>8){window.fbSelectedBox=b;document.getElementById('fb-result').textContent='Selected: '+b.map(Math.round).join(', ')}};
    })();
    </script>
    """.replace("__IMAGE__",dat)
    display(HTML(html))
    MANUAL_BOX=eval_js("new Promise(resolve => {const t=setInterval(()=>{if(window.fbSelectedBox){clearInterval(t);resolve(window.fbSelectedBox)}},200)})")
anchor_box=clip_box(MANUAL_BOX)
if anchor_box is None:raise ValueError("Set a valid MANUAL_BOX")
print("Anchor frame",MANUAL_FRAME,"ROI",anchor_box)

## 7. Pose・YOLO候補の前計算

MediaPipe VIDEOモードは時刻順に実行し、その後に手動アンカーから双方向追跡します。

In [ ]:
pose_cache={};yolo_cache={}
cap=cv2.VideoCapture(str(source))
for i in tqdm(range(total),desc='Pose + YOLO precompute'):
    ok,img=cap.read()
    if not ok:break
    arm=pose_arm(img,int(round(i*1000/fps)) if fps>0 else i*33)
    pose_cache[i]=arm
    yolo_cache[i]=detect_hands(img)
cap.release();pose.close()
if len(pose_cache)!=total:raise RuntimeError('Some frames could not be decoded')
print('Pose valid:',sum(x is not None for x in pose_cache.values()))
print('YOLO candidates:',sum(bool(v[0]) for v in yolo_cache.values()))

## 8. アンカーを起点とした前後両方向の追跡

検出と光学フローの品質を別々に記録します。投球手を見失った後、別の位置のYOLO候補へ飛び移らないようにします。

In [ ]:
TRACK_GATE=1.45
def read_at(cap,i):
    cap.set(cv2.CAP_PROP_POS_FRAMES,int(i))
    ok,img=cap.read()
    if not ok:raise RuntimeError('Frame read failed '+str(i))
    return img
def run_direction(indices):
    cap=cv2.VideoCapture(str(source))
    prev_gray=None;box=anchor_box.copy();age=0;out={}
    for i in tqdm(indices,desc='Tracking'):
        img=read_at(cap,i)
        gray=cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
        arm=pose_cache[i]
        candidates,maxconf=yolo_cache[i]
        if i==MANUAL_FRAME:
            out[i]={'box':anchor_box.copy(),'source':'manual','pose_valid':int(arm is not None),
                    'score':np.nan,'flow_fraction':np.nan,'max_conf':maxconf,'age':0}
            prev_gray=gray
            continue
        predicted,flow_fraction=flow_box(prev_gray,gray,box)
        ref=predicted if predicted is not None else (box if age<=FLOW_MAX_AGE else None)
        found=None;score=np.nan;origin='none'
        if ref is not None:
            pool=[]
            for b,conf in candidates:
                dist=np.linalg.norm(ctr(b)-ctr(ref))/max(boxside(ref),15)
                if dist>TRACK_GATE:continue
                if arm is not None:
                    wrist_dist=np.linalg.norm(ctr(b)-arm['wrist'])/max(arm['forearm'],1)
                    if wrist_dist>WRIST_GATE:continue
                else:wrist_dist=0.
                pool.append((conf-.25*dist-.15*wrist_dist,b,conf))
            if pool:
                _,found,score=max(pool,key=lambda x:x[0])
                origin='yolo_pose' if arm is not None else 'yolo_track'
        if found is not None:
            box=found.copy();age=0
        elif predicted is not None and age<FLOW_MAX_AGE:
            box=predicted.copy();age+=1;origin='flow'
        else:
            box=None;age+=1;origin='none'
        out[i]={'box':box.copy() if box is not None else None,'source':origin,
                'pose_valid':int(arm is not None),'score':float(score),
                'flow_fraction':float(flow_fraction),'max_conf':maxconf,'age':age}
        prev_gray=gray
    cap.release()
    return out
forward=run_direction(list(range(MANUAL_FRAME,total)))
backward=run_direction(list(range(MANUAL_FRAME,-1,-1)))
tracking={**backward,**forward}
assert len(tracking)==total
print(pd.Series([tracking[i]['source'] for i in range(total)]).value_counts())